In [1]:

import pandas as pd
from IPython.display import display, HTML

# ============================================================
# 1. CONFIGURAÇÃO DOS ARQUIVOS
# ============================================================

arquivo_dados = "01_PENSE2024_MICRODADOS.parquet"

arquivo_dicionario = "00_Dicionário_Microdados_PENSE2024.xlsx"

arquivo_saida = "01_Analise_Respostas_Especiais_PENSE2024.xlsx"

# Percentual mínimo para selecionar uma variável
limite_percentual = 5

# Códigos especiais
codigos_especiais = [-9, -2, -1]


# ============================================================
# 2. CARREGAR OS ARQUIVOS
# ============================================================

print("Carregando base de dados...")

df = pd.read_parquet(arquivo_dados)

print("Carregando dicionário...")

dicionario = pd.read_excel(arquivo_dicionario)

print(f"Total de registros: {len(df):,}")
print(f"Total de variáveis: {len(df.columns):,}")


# ============================================================
# 3. PADRONIZAR OS NOMES DAS VARIÁVEIS
# ============================================================

df.columns = df.columns.astype(str).str.strip().str.upper()

dicionario.columns = dicionario.columns.str.strip()

dicionario["Nome da Variável"] = (
    dicionario["Nome da Variável"]
    .astype(str)
    .str.strip()
    .str.upper()
)

# Remover eventuais espaços extras das descrições
dicionario["Descrição da Variável"] = (
    dicionario["Descrição da Variável"]
    .fillna("Descrição não disponível")
    .astype(str)
    .str.strip()
)


# ============================================================
# 4. PREPARAR O DICIONÁRIO
# ============================================================

# Manter somente as colunas necessárias
dicionario_reduzido = dicionario[
    [
        "Nome da Variável",
        "Descrição da Variável"
    ]
].drop_duplicates(subset="Nome da Variável")


# ============================================================
# 5. ANALISAR OS CÓDIGOS ESPECIAIS
# ============================================================

total_linhas = len(df)

resultados = []

print("\nAnalisando as variáveis...")

for coluna in df.columns:

    # Converter valores numéricos
    valores = pd.to_numeric(
        df[coluna],
        errors="coerce"
    )

    # Contar ocorrências
    qtd_menos9 = (valores == -9).sum()

    qtd_menos2 = (valores == -2).sum()

    qtd_menos1 = (valores == -1).sum()

    # Total de códigos especiais
    total_especiais = (
        qtd_menos9 +
        qtd_menos2 +
        qtd_menos1
    )

    # Calcular percentual
    percentual = (
        total_especiais / total_linhas
    ) * 100

    # Selecionar variáveis acima do limite
    if percentual > limite_percentual:

        resultados.append({

            "Código": coluna,

            "Quantidade -9": qtd_menos9,

            "Quantidade -2": qtd_menos2,

            "Quantidade -1": qtd_menos1,

            "Total códigos especiais": total_especiais,

            "Total de registros": total_linhas,

            "Percentual (%)": round(percentual, 2)

        })


# ============================================================
# 6. CRIAR DATAFRAME DOS RESULTADOS
# ============================================================

resultado_df = pd.DataFrame(resultados)

if resultado_df.empty:

    print(
        f"Nenhuma variável apresentou mais de "
        f"{limite_percentual}% de códigos especiais."
    )

else:

    # Ordenar pelo percentual decrescente
    resultado_df = resultado_df.sort_values(
        by="Percentual (%)",
        ascending=False
    )

    # Associar as perguntas do dicionário
    resultado_df = resultado_df.merge(
        dicionario_reduzido,
        left_on="Código",
        right_on="Nome da Variável",
        how="left"
    )

    # Remover coluna duplicada
    resultado_df = resultado_df.drop(
        columns=["Nome da Variável"]
    )

    # Tratar descrições não encontradas
    resultado_df["Descrição da Variável"] = (
        resultado_df["Descrição da Variável"]
        .fillna("Descrição não encontrada no dicionário")
    )


    # ========================================================
    # 7. FORMATAR A DISTRIBUIÇÃO DOS CÓDIGOS
    # ========================================================

    def formatar_distribuicao(linha):

        total = linha["Total de registros"]

        distribuicao = []

        for codigo, coluna in [
            ("-9", "Quantidade -9"),
            ("-2", "Quantidade -2"),
            ("-1", "Quantidade -1")
        ]:

            quantidade = linha[coluna]

            percentual = (
                quantidade / total
            ) * 100

            if quantidade > 0:

                distribuicao.append(
                    f"[{codigo}] "
                    f"{quantidade:,} ocorrências: "
                    f"{percentual:.2f}%"
                )

        return "<br>".join(distribuicao)


    resultado_df["Distribuição das Respostas"] = (
        resultado_df.apply(
            formatar_distribuicao,
            axis=1
        )
    )


    # ========================================================
    # 8. MONTAR A TABELA VISUAL
    # ========================================================

    resultado_df["Pergunta / Variável"] = (
        "<b>"
        + resultado_df["Código"]
        + "</b><br><br>"
        + resultado_df["Descrição da Variável"]
    )

    tabela_final = resultado_df[
        [
            "Pergunta / Variável",
            "Distribuição das Respostas"
        ]
    ].copy()

    # ========================================================
    # 9. EXIBIR NO JUPYTER NOTEBOOK
    # ========================================================

    print("\n========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========")

    display(HTML(
        tabela_final.to_html(
            index=True,
            escape=False,
            justify="left",
            border=0,
            classes="tabela-pense"
        )
    ))

    # ========================================================
    # 10. EXPORTAR RESULTADOS PARA EXCEL
    # ========================================================

    # Versão detalhada para análise posterior
    colunas_excel = [
        "Código",
        "Descrição da Variável",
        "Quantidade -9",
        "Quantidade -2",
        "Quantidade -1",
        "Total códigos especiais",
        "Total de registros",
        "Percentual (%)"
    ]

    resultado_excel = resultado_df[colunas_excel].copy()

    with pd.ExcelWriter(
        arquivo_saida,
        engine="openpyxl"
    ) as writer:

        resultado_excel.to_excel(
            writer,
            sheet_name="Analise Detalhada",
            index=False
        )

        tabela_final.to_excel(
            writer,
            sheet_name="Visualizacao",
            index=False
        )

        # Ajustar largura das colunas
        for planilha in writer.sheets.values():

            planilha.freeze_panes = "A2"

            for coluna in planilha.columns:

                letra = coluna[0].column_letter

                largura = max(
                    len(str(celula.value or ""))
                    for celula in coluna
                )

                planilha.column_dimensions[letra].width = min(
                    max(largura + 2, 15),
                    60
                )

    print("\n===================================================")

    print(f"Variáveis identificadas: {len(resultado_df)}")

    print(f"Arquivo Excel salvo: {arquivo_saida}")

    print("Análise concluída com sucesso!")


Carregando base de dados...
Carregando dicionário...
Total de registros: 153,870
Total de variáveis: 430

Analisando as variáveis...

========== EXIBIÇÃO DOS RESULTADOS NO IPYNB ==========


,Pergunta / Variável,Distribuição das Respostas
0,E01P10107Quais foram as dificuldades encontradas para a aquisição de alimentos da agricultura familiar utilizados na preparação da comida/merenda escolar? - Não sabe / não lembra,"[-1] 144,696 ocorrências: 94.04%"
1,E01P10102Quais foram as dificuldades encontradas para a aquisição de alimentos da agricultura familiar utilizados na preparação da comida/merenda escolar? - A quantidade de alimentos que o fornecedor tinha disponível era insuficiente para atender as necessidades da escola.,"[-1] 144,696 ocorrências: 94.04%"
2,E01P10105Quais foram as dificuldades encontradas para a aquisição de alimentos da agricultura familiar utilizados na preparação da comida/merenda escolar? - Desconhecimento sobre os trâmites necessários para adquirir os alimentos da agricultura familiar,"[-1] 144,696 ocorrências: 94.04%"
3,E01P10101Quais foram as dificuldades encontradas para a aquisição de alimentos da agricultura familiar utilizados na preparação da comida/merenda escolar? - Atraso na entrega dos alimentos,"[-1] 144,696 ocorrências: 94.04%"
4,E01P10103Quais foram as dificuldades encontradas para a aquisição de alimentos da agricultura familiar utilizados na preparação da comida/merenda escolar? - Desconhecimento da existência de fornecedores de alimentos da agricultura familiar em localidades próximas à escola,"[-1] 144,696 ocorrências: 94.04%"
5,"E01P10104Quais foram as dificuldades encontradas para a aquisição de alimentos da agricultura familiar utilizados na preparação da comida/merenda escolar? - Fornecedor não apresentou a documentação exigida (cadastros DPF e/ou CAF atualizados, projeto de vendas etc.)","[-1] 144,696 ocorrências: 94.04%"
6,E01P10106Quais foram as dificuldades encontradas para a aquisição de alimentos da agricultura familiar utilizados na preparação da comida/merenda escolar? - Outras,"[-1] 144,696 ocorrências: 94.04%"
7,"B04009BNOS ÚLTIMOS 30 DIAS, na maioria das vezes, como você conseguiu seus próprios cigarros?","[-9] 33 ocorrências: 0.02%[-2] 133 ocorrências: 0.09%[-1] 142,544 ocorrências: 92.64%"
8,B06008Qual dessas drogas você usou pela primeira vez?,"[-9] 128 ocorrências: 0.08%[-2] 179 ocorrências: 0.12%[-1] 139,226 ocorrências: 90.48%"
9,B0600704Você já experimentou alguma droga desses grupos? - Outra,"[-9] 121 ocorrências: 0.08%[-2] 179 ocorrências: 0.12%[-1] 139,226 ocorrências: 90.48%"



Variáveis identificadas: 210
Arquivo Excel salvo: 01_Analise_Respostas_Especiais_PENSE2024.xlsx
Análise concluída com sucesso!
